In [ ]:
# ======================================
# 1D CNN FOR DIABETES PREDICTION
# Architecture: Conv1D → BatchNorm → Dropout → Dense
# Loss: Focal Loss
# Strategy: High Recall Focus
# ======================================

# ======================================
# 1. IMPORTS
# ======================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras import backend as K

from itertools import combinations
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, precision_recall_curve
)
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import EditedNearestNeighbours

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)


# ======================================
# 2. FOCAL LOSS FOR KERAS
# ======================================
def focal_loss(alpha=0.75, gamma=2.0):
    """
    Focal Loss for imbalanced classification
    - alpha: weight for positive class
    - gamma: focusing parameter
    """
    def focal_loss_fixed(y_true, y_pred):
        # Cast y_true to float32 to match y_pred
        y_true = tf.cast(y_true, tf.float32)
        y_pred = K.clip(y_pred, K.epsilon(), 1 - K.epsilon())
        
        # Focal loss calculation
        pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = K.pow(1 - pt, gamma)
        
        # Binary cross entropy
        bce = -y_true * K.log(y_pred) - (1 - y_true) * K.log(1 - y_pred)
        
        # Apply focal weight and alpha
        loss = alpha * focal_weight * bce
        
        return K.mean(loss)
    
    return focal_loss_fixed


# ======================================
# 3. CUSTOM METRICS
# ======================================
def recall_metric(y_true, y_pred):
    """Recall metric for monitoring"""
    # Cast y_true to float32 to match y_pred
    y_true = tf.cast(y_true, tf.float32)
    y_pred_binary = K.round(y_pred)
    true_positives = K.sum(K.round(K.clip(y_true * y_pred_binary, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall

def precision_metric(y_true, y_pred):
    """Precision metric for monitoring"""
    # Cast y_true to float32 to match y_pred
    y_true = tf.cast(y_true, tf.float32)
    y_pred_binary = K.round(y_pred)
    true_positives = K.sum(K.round(K.clip(y_true * y_pred_binary, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred_binary, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision


# ======================================
# 4. LOAD DATA
# ======================================
df = pd.read_csv("/kaggle/input/diabd-cohort/DiaBD_A Diabetes Dataset for Enhanced Risk Analysis and Research in Bangladesh.csv")
print("="*70)
print("1D CNN WITH FOCAL LOSS - DIABETES PREDICTION")
print("="*70)
print(f"\nOriginal shape: {df.shape}")


# ======================================
# 5. ENCODING
# ======================================
print("\n" + "="*70)
print("STEP 1: ENCODING")
print("="*70)

cat_cols = df.select_dtypes(include=['object']).columns.tolist()
le = LabelEncoder()
df['diabetic'] = le.fit_transform(df['diabetic'])
cat_cols = [c for c in cat_cols if c != 'diabetic']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(f"After encoding: {df.shape}")


# ======================================
# 6. FEATURE ENGINEERING
# ======================================
print("\n" + "="*70)
print("STEP 2: FEATURE ENGINEERING")
print("="*70)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols.remove('diabetic')

print("Creating interaction features...")
for f1, f2 in combinations(numeric_cols, 2):
    df[f'{f1}_x_{f2}'] = df[f1] * df[f2]

print("Creating ratio features...")
for f1, f2 in combinations(numeric_cols, 2):
    df[f'{f1}_div_{f2}'] = df[f1] / (df[f2] + 1e-6)

print("Creating polynomial features...")
for col in numeric_cols:
    df[f'{col}_squared'] = df[col] ** 2

print(f"After feature engineering: {df.shape}")


# ======================================
# 7. ENN CLEANING
# ======================================
print("\n" + "="*70)
print("STEP 3: ENN CLEANING")
print("="*70)

X = df.drop('diabetic', axis=1)
y = df['diabetic']

print("\nOriginal class distribution:")
print(y.value_counts().to_dict())

enn = EditedNearestNeighbours(n_neighbors=3)
X_enn, y_enn = enn.fit_resample(X, y)

print("\nAfter ENN cleaning:")
print(pd.Series(y_enn).value_counts().to_dict())
print(f"Samples removed: {len(y) - len(y_enn)}")


# ======================================
# 8. TRAIN-TEST SPLIT
# ======================================
print("\n" + "="*70)
print("STEP 4: TRAIN-TEST SPLIT")
print("="*70)

X_train, X_test, y_train, y_test = train_test_split(
    X_enn, y_enn, test_size=0.2, random_state=42, stratify=y_enn
)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print("\nTrain distribution:")
print(pd.Series(y_train).value_counts().to_dict())


# ======================================
# 9. SMOTE OVERSAMPLING
# ======================================
print("\n" + "="*70)
print("STEP 5: BORDERLINE-SMOTE OVERSAMPLING")
print("="*70)

n_train_positive = (y_train == 1).sum()
n_train_negative = (y_train == 0).sum()

# Target 3x oversampling
target_positive = n_train_positive * 3
sampling_ratio = min(target_positive / n_train_negative, 1.0)

borderline_smote = BorderlineSMOTE(
    sampling_strategy=sampling_ratio,
    k_neighbors=5,
    random_state=42,
    kind='borderline-1'
)

X_train_array = X_train.values if hasattr(X_train, 'values') else X_train
y_train_array = y_train.values if hasattr(y_train, 'values') else y_train

X_resampled, y_resampled = borderline_smote.fit_resample(X_train_array, y_train_array)

print("\nAfter SMOTE:")
print(pd.Series(y_resampled).value_counts().to_dict())


# ======================================
# 10. FEATURE SCALING
# ======================================
print("\n" + "="*70)
print("STEP 6: FEATURE SCALING")
print("="*70)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_resampled)
X_test_scaled = scaler.transform(X_test)

# Reshape for CNN (add channel dimension)
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

print(f"Train shape for CNN: {X_train_cnn.shape}")
print(f"Test shape for CNN: {X_test_cnn.shape}")


# ======================================
# 11. BUILD 1D CNN MODEL
# ======================================
print("\n" + "="*70)
print("STEP 7: BUILD 1D CNN MODEL")
print("="*70)

def build_cnn_model(input_shape):
    """
    1D CNN Architecture for tabular data
    """
    model = models.Sequential([
        # First Conv Block
        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same', 
                     input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Second Conv Block
        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.3),
        
        # Third Conv Block
        layers.Conv1D(32, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Global pooling
        layers.GlobalAveragePooling1D(),
        
        # Dense layers
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        
        # Output layer
        layers.Dense(1, activation='sigmoid')
    ])
    
    return model

model = build_cnn_model((X_train_cnn.shape[1], 1))

# Compile with focal loss
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=focal_loss(alpha=0.75, gamma=2.0),
    metrics=['accuracy', recall_metric, precision_metric, keras.metrics.AUC(name='auc')]
)

print("\n" + "="*70)
print("MODEL ARCHITECTURE")
print("="*70)
model.summary()


# ======================================
# 12. CALLBACKS
# ======================================
print("\n" + "="*70)
print("STEP 8: SETUP CALLBACKS")
print("="*70)

# Early stopping
early_stop = callbacks.EarlyStopping(
    monitor='val_recall_metric',
    patience=20,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_recall_metric',
    factor=0.5,
    patience=10,
    mode='max',
    min_lr=1e-6,
    verbose=1
)

# Model checkpoint
checkpoint = callbacks.ModelCheckpoint(
    'best_cnn_model.h5',
    monitor='val_recall_metric',
    mode='max',
    save_best_only=True,
    verbose=0
)

print("Callbacks configured: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint")


# ======================================
# 13. TRAIN MODEL
# ======================================
print("\n" + "="*70)
print("STEP 9: TRAINING CNN MODEL")
print("="*70)

# Class weights for additional emphasis
class_weight = {
    0: 1.0,
    1: (y_resampled == 0).sum() / (y_resampled == 1).sum()
}

print(f"Class weights: {class_weight}")

history = model.fit(
    X_train_cnn, y_resampled,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

print("\n✓ Training completed!")


# ======================================
# 14. PREDICTIONS
# ======================================
print("\n" + "="*70)
print("STEP 10: PREDICTIONS")
print("="*70)

# Get probabilities
y_pred_proba = model.predict(X_test_cnn).flatten()

# Find optimal threshold using F1 score
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)
precisions = precisions[:-1]
recalls = recalls[:-1]

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-6)
valid_mask = ~np.isnan(f1_scores) & ~np.isinf(f1_scores)

if valid_mask.sum() > 0:
    valid_f1 = f1_scores[valid_mask]
    valid_thresholds = thresholds[valid_mask]
    best_idx = np.argmax(valid_f1)
    best_threshold = valid_thresholds[best_idx]
else:
    best_threshold = 0.5

print(f"Optimal threshold: {best_threshold:.3f}")

y_pred = (y_pred_proba >= best_threshold).astype(int)


# ======================================
# 15. EVALUATION
# ======================================
print("\n" + "="*70)
print("STEP 11: EVALUATION METRICS")
print("="*70)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1-score', 'Precision', 'Recall', 'ROC-AUC'],
    'Value': [accuracy, f1, precision, recall, auc]
})

print("\n" + "="*70)
print("TEST SET METRICS")
print("="*70)
print(metrics_df.to_string(index=False))


# ======================================
# 16. CONFUSION MATRIX
# ======================================
cm = confusion_matrix(y_test, y_pred)

print("\n" + "="*70)
print("CONFUSION MATRIX")
print("="*70)
print(cm)
print("\nDetailed Breakdown:")
print(f"  True Negatives (TN):  {cm[0, 0]:4d}")
print(f"  False Positives (FP): {cm[0, 1]:4d}")
print(f"  False Negatives (FN): {cm[1, 0]:4d} ⚠️ MINIMIZE THIS")
print(f"  True Positives (TP):  {cm[1, 1]:4d} ✓")

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=['Non-Diabetic', 'Diabetic'],
            yticklabels=['Non-Diabetic', 'Diabetic'],
            cbar_kws={'label': 'Count'},
            annot_kws={'size': 16})
plt.title('Confusion Matrix - 1D CNN with Focal Loss',
          fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()


# ======================================
# 17. TRAINING HISTORY
# ======================================
print("\n" + "="*70)
print("TRAINING HISTORY VISUALIZATION")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train Loss')
axes[0, 0].plot(history.history['val_loss'], label='Val Loss')
axes[0, 0].set_title('Model Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Recall
axes[0, 1].plot(history.history['recall_metric'], label='Train Recall')
axes[0, 1].plot(history.history['val_recall_metric'], label='Val Recall')
axes[0, 1].set_title('Recall')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Recall')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Precision
axes[1, 0].plot(history.history['precision_metric'], label='Train Precision')
axes[1, 0].plot(history.history['val_precision_metric'], label='Val Precision')
axes[1, 0].set_title('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True)

# AUC
axes[1, 1].plot(history.history['auc'], label='Train AUC')
axes[1, 1].plot(history.history['val_auc'], label='Val AUC')
axes[1, 1].set_title('ROC-AUC')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('AUC')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()


# ======================================
# 18. FINAL SUMMARY
# ======================================
print("\n" + "="*70)
print("PIPELINE COMPLETED!")
print("="*70)
print("\n✅ 1D CNN Implementation:")
print("   1. ✓ ENN Cleaning")
print("   2. ✓ BorderlineSMOTE Oversampling")
print("   3. ✓ Feature Scaling")
print("   4. ✓ 1D CNN Architecture (Conv1D + BatchNorm + Dropout)")
print("   5. ✓ Focal Loss Function")
print("   6. ✓ Class Weighting")
print("   7. ✓ Early Stopping & Learning Rate Scheduling")
print("   8. ✓ Optimal Threshold Selection")
print("\n🎯 Goal: High Recall with CNN's pattern recognition capability")
print("="*70)